# Building a custom robot config for cuRobo

cuRobo only ships a handful of example robot configs by name (`franka.yml`, `ur10e.yml`, `dual_ur10e.yml`, `unitree_g1.yml`, ...). If your robot isn't one of these, you need to build your own config from a URDF - including fitting collision spheres yourself, since cuRobo represents robot links as spheres for fast GPU collision checking.

cuRobo v2 ships a first-party `RobotBuilder` (`curobo.robot_builder`) for this, replacing the old external "bubblify" tool from cuRobo v1. This notebook walks through the whole pipeline, first with cuRobo's bundled `ur10e.urdf` as a minimal single-URDF example, then with a more realistic combined URDF - a Realman RM75-6F arm with a Schunk EGK40 Magneto gripper and a wrist-mounted RealSense D435 camera, all from [`airo-models`](https://github.com/airo-ugent/airo-models) - as the concrete, runnable end-to-end example. The same steps apply to any URDF of your own.


## 1. Locate your URDF and mesh assets

`RobotBuilder` resolves paths relative to cuRobo's own installed assets directory (`curobo.content.get_assets_path()`). If your URDF and meshes live *inside* that directory tree, the saved config stores portable relative paths; otherwise it falls back to absolute paths (which still work, just aren't portable to another machine).

For a robot you're adding to your own cuRobo checkout, the natural place to put it is `<curobo>/curobo/content/assets/robot/<your_robot>/`, alongside the existing bundled robots (e.g. `ur_description/`, `franka_description/`).

**If your robot is one that [`airo-models`](https://github.com/airo-ugent/airo-models) already ships** (UR3e, UR5e, Robotiq 2F-85, ...), use that URDF directly instead of hand-rolling your own - it already has correctly-resolving meshes, unlike a from-scratch URDF where you have to get mesh paths right yourself (see the next cell for exactly why that matters).

In [ ]:
from pathlib import Path

from curobo.content import get_assets_path

# Substitute your own URDF + asset directory here. This example reuses cuRobo's bundled UR10e
# so the notebook is runnable out of the box.
urdf_path = Path(get_assets_path()) / "robot/ur_description/ur10e.urdf"
asset_path = Path(get_assets_path()) / "robot/ur_description"
assert urdf_path.exists(), f"URDF not found: {urdf_path}"

# If your URDF references mesh files (e.g. <mesh filename="visual/forearm.obj"/>), those files must
# actually exist under asset_path - RobotBuilder will fail to load meshes otherwise. This bit us
# directly: a UR3e URDF placed under ur_description/ referenced meshes that were never added
# alongside it (only ur5e/ and ur10e/ mesh subdirectories exist there), so building it failed.
# Double check with e.g. `grep -o 'filename="[^"]*"' your.urdf` before proceeding.

### Using an `airo-models` URDF instead

`airo_models.get_urdf_path(name)` returns the absolute path to a bundled URDF (e.g. `"ur3e"`, `"rm75_6f"`, `"schunk_egk40_magneto"`). Unlike cuRobo's own robots (which keep meshes in a shared `meshes/<robot>/visual/...` subdirectory), `airo-models` keeps each URDF next to its own `visual/`/`collision/` mesh folders - so the asset path `RobotBuilder` needs is simply the URDF's own parent directory. This was verified end-to-end (mesh loading, sphere fitting, self-collision matrix, and actual planning with `SingleArmCuroboPlanner`) against `airo_models`' bundled Realman RM75-6F arm:

In [ ]:
import os

import airo_models
print("Available airo_models:", airo_models.AIRO_MODEL_NAMES)


airo_models_urdf_path = airo_models.get_urdf_path("rm75_6f")
airo_models_asset_path = os.path.dirname(airo_models_urdf_path)  # URDF's own directory, not a "meshes/" subfolder

print(airo_models_urdf_path)
urdf_path = airo_models_urdf_path
asset_path = airo_models_asset_path


### Combining arm + gripper + camera into a single URDF

`RobotBuilder` takes one URDF. If your robot is an arm with an attached gripper and/or a wrist camera, you need to merge them before building the sphere config.

The steps below combine the RM75-6F arm with the Schunk EGK40 Magneto gripper (parallel gripper with magnetic fingertips) and a wrist-mounted RealSense D435, using the same `attach_urdf` calls and mounting values as `airo-models`' own `scripts/combine_urdf_example.py`:
1. Load each component URDF from `airo_models` and make all mesh paths absolute (so the merged file resolves them regardless of where it's written).
2. Freeze gripper and camera joints with `make_static` – cuRobo plans only the arm's DOF; the gripper and camera are treated as rigid attached links.
3. Prefix every link/joint name (`schunk_`, `d435_`) to avoid name collisions in the merged URDF.
4. Merge all links and joints, then add three fixed attaching joints:
   - arm `tool0` → `schunk_base_link` (gripper at flange)
   - arm `tool0` → `d435_base_link` (camera on wrist – **adjust the offset to match your physical bracket**)
   - `schunk_base_link` → `schunk_tcp` (virtual tip between finger mounts, used as planning target)


In [ ]:
import airo_models
from airo_models import urdf as urdf_utils
from airo_models.urdf import attach_urdf

# --- Load each component ---
arm_urdf_path = airo_models.get_urdf_path("rm75_6f")
gripper_urdf_path = airo_models.get_urdf_path("schunk_egk40_magneto")
camera_urdf_path = airo_models.get_urdf_path("d435")

arm_dict = urdf_utils.read_urdf(str(arm_urdf_path))
urdf_utils.make_paths_absolute(arm_dict, str(arm_urdf_path))

gripper_dict = urdf_utils.read_urdf(str(gripper_urdf_path))
camera_dict = urdf_utils.read_urdf(str(camera_urdf_path))

# Attach the gripper to the arm flange. attachment_urdf_path makes mesh paths absolute before
# merging. freeze_joints=True (default) converts gripper joints to fixed - cuRobo plans only
# the arm's DOF.
attach_urdf(arm_dict, gripper_dict, parent_link="tool0", child_prefix="schunk_",
            attachment_urdf_path=str(gripper_urdf_path))

# Wrist camera: 5 cm along tool0 y-axis, tilted 30° (π/6) downward around the x-axis so the
# optical axis (camera Z+) points toward the gripper fingertips - mounting values reused as-is
# from airo-models' combine_urdf_example.py.
attach_urdf(arm_dict, camera_dict, parent_link="tool0", child_prefix="d435_",
            attachment_urdf_path=str(camera_urdf_path),
            origin_xyz="0.02 0.05 0.06", origin_rpy="-0.3 0 3.14")  # ← adjust xyz/rpy for your physical bracket

# Virtual TCP at the tip of the gripper body (between finger mounts); used as the planning target.
arm_dict["robot"]["link"].append({"@name": "schunk_tcp"})
arm_dict["robot"]["joint"].append({
    "@name": "gripper_to_tcp",
    "@type": "fixed",
    "parent": {"@link": "schunk_base_link"},
    "child": {"@link": "schunk_tcp"},
    "origin": {"@xyz": "0 0 0.0835", "@rpy": "0 0 0"},
})

# --- Write combined URDF ---
combined_urdf_path = "/tmp/rm75_6f_schunk_magneto_d435.urdf"
urdf_utils.write_urdf_to_file(arm_dict, combined_urdf_path)
urdf_path = combined_urdf_path
asset_path = ""  # all mesh paths are now absolute
print(f"Combined URDF written to: {combined_urdf_path}")


### Gotcha: links with mass but no `<inertia>` tensor crash the parser

Several links in the Magneto gripper URDF (`base_link`, `camera_mount_plate`, `pcb_mount_plate`, `base_link_gripper`, both fingers) declare `<inertial><mass .../></inertial>` with a mass but **no `<inertia>` tensor**. cuRobo's URDF parser (via `yourdfpy`) treats "no `<inertial>` element at all" and "`<inertial>` present but missing `<inertia>`" very differently: the former falls back to a small default inertia, but the latter leaves `inertial.inertia = None` and crashes later with `TypeError: 'NoneType' object is not subscriptable` inside `RobotBuilder.compute_collision_matrix()`. This is real, reproducible - not a hypothetical - and isn't specific to `RobotBuilder`; any cuRobo pipeline that parses this URDF hits it.

The fix: patch a small placeholder `<inertia>` tensor onto every link that has `<inertial>` but is missing it, before handing the merged URDF to `RobotBuilder`.

In [ ]:
def patch_missing_inertia_tensors(urdf_dict: dict) -> int:
    """Add a small placeholder <inertia> tensor to every link with <inertial> but no <inertia>.

    Returns the number of links patched.
    """
    links = urdf_dict["robot"]["link"]
    if isinstance(links, dict):
        links = [links]
    n_patched = 0
    for link in links:
        inertial = link.get("inertial")
        if inertial is not None and "inertia" not in inertial:
            inertial["inertia"] = {
                "@ixx": "1e-4", "@iyy": "1e-4", "@izz": "1e-4",
                "@ixy": "0", "@ixz": "0", "@iyz": "0",
            }
            n_patched += 1
    return n_patched


n_patched = patch_missing_inertia_tensors(arm_dict)
print(f"Patched missing <inertia> tensors on {n_patched} links")
urdf_utils.write_urdf_to_file(arm_dict, combined_urdf_path)  # re-write with the patched inertials


## 2. Create the builder and fit collision spheres

`fit_collision_spheres` approximates each link's mesh with a set of spheres. We pass `use_collision_mesh=True` so the spheres are fit to each link's `<collision>` geometry rather than the (default) `<visual>` geometry: the collision meshes are the simplified, convex-decomposed hulls intended for collision checking, so they give tighter, cheaper sphere fits and match the geometry the planner actually reasons about. `sphere_density` controls how many spheres per link (practical range 0.1-10.0); `compute_metrics=True` reports fit quality (`coverage` close to 1.0 is good, high `protrusion` means spheres bulge outside the mesh).

We also clip the **first link that has a `<collision>` element** at `z=0` via `clip_links`. That link is the robot's mounting base, so any spheres poking below its `z=0` mounting plane would spuriously collide with the surface the robot is mounted on. We skip pure virtual/inertia-only links (no collision geometry) since `RobotBuilder` ignores them anyway.


In [ ]:
from curobo.robot_builder import RobotBuilder

from airo_models import urdf as urdf_utils

# Clip the first link that has a <collision> element at z=0 so its spheres don't dip below
# the mounting plane and spuriously collide with the surface the robot stands on.
# (Pure virtual/inertia-only links have no collision geometry and nothing to clip.)
links = urdf_utils.read_urdf(str(urdf_path))["robot"]["link"]
first_collision_link = next(l["@name"] for l in links if "collision" in l)
print(f"First link with collision geometry: {first_collision_link}")

builder = RobotBuilder(str(urdf_path), str(asset_path), tool_frames=["schunk_tcp"])
builder.fit_collision_spheres(
    sphere_density=3.0,
    use_collision_mesh=True,
    clip_links={first_collision_link: ("z", 0.0)},
    compute_metrics=True,
)

print(f"Fitted {builder.num_spheres} spheres across {len(builder.collision_link_names)} links")
for link, metrics in builder.link_metrics.items():
    print(f"  {link}: {metrics.num_spheres} spheres, coverage={metrics.coverage:.2f}")


If a specific link's fit is bad, refit just that link instead of the whole robot:
```python
builder.refit_link_spheres("link_6", sphere_density=2.0)
```
or clip spheres away from one side of a link (e.g. to avoid the base link's spheres poking below the mounting plate):
```python
builder.fit_collision_spheres(clip_links={"base_link": ("z", 0.0)}, use_collision_mesh=True)
```

In [ ]:
builder.refit_link_spheres("schunk_right_magneto_finger",sphere_density=10.0)
builder.refit_link_spheres("schunk_left_magneto_finger",sphere_density=10.0)

### Optional: inspect the fit interactively

`builder.visualize()` opens an interactive Viser viewer so you can see the fitted spheres against the robot. **It blocks the cell** until `timeout_sec` elapses (or forever if you don't pass one) - open the printed URL in your browser, then either wait for the timeout or interrupt the cell when you're done looking.


In [ ]:
builder.visualize(timeout_sec=30)

## 3. Compute the self-collision ignore matrix

This figures out which link pairs should never be checked against each other: neighboring links (always ignored), links that collide at the *current default joint configuration* (treated as false positives), and - optionally - pairs sampled to never collide across many random configurations (pruned for efficiency).

⚠️ **Gotcha:** the "collide at default configuration" check uses whatever the robot's current default joint position is *at this point in the pipeline* - which is all-zeros unless you've loaded an existing config with `RobotBuilder.from_config(...)`. If you plan to use a different "home"/retract pose later, false positives at *that* pose won't be caught here - verify separately with `RobotDebugger` in step 5.

In [ ]:
builder.compute_collision_matrix(num_samples=1000)
print(f"{len(builder.collision_matrix)} links have ignore entries")

## 4. Build and save

⚠️ **Gotcha:** `builder.save(...)` writes two internal loader flags (`load_collision_spheres`, `num_envs`) into the YAML's `kinematics:` block. When you later load that same file the normal way (e.g. `MotionPlannerCfg.create(robot="my_robot.yml")`, which is what `SingleArmCuroboPlanner` does), cuRobo's loader passes those same two flags explicitly *and* unpacks the YAML's own `kinematics:` dict as keyword arguments - causing a `TypeError: got multiple values for keyword argument 'load_collision_spheres'`. Strip both keys after saving; the file works fine without them.

In [ ]:
import yaml

config = builder.build()
output_path = "/tmp/rm75_6f_schunk_magneto_d435.yml"
builder.save(config, output_path)

# Work around the load_collision_spheres/num_envs bug described above.
with open(output_path) as f:
    data = yaml.safe_load(f)
data["kinematics"].pop("load_collision_spheres", None)
data["kinematics"].pop("num_envs", None)
with open(output_path, "w") as f:
    yaml.safe_dump(data, f)

## 5. Verify self-collision at your chosen default pose

`RobotDebugger` checks *self*-collision only (not collision with the world/scene). Since `compute_collision_matrix()` (step 3) only auto-suppressed false positives at the all-zeros pose, explicitly check whichever pose you actually intend to use as "home".

In [ ]:
from curobo.robot_builder import RobotDebugger

debugger = RobotDebugger(output_path)
result = debugger.check_default_joint_configuration_collision()
print("Self-collision at default pose:", result["has_collision"])

# To check a specific non-default "home" pose instead:
# result = debugger.check_collision_at_config([0.0, -2.2, 1.9, -1.383, -1.57, 0.0])

## 6. Use it with `SingleArmCuroboPlanner`

This is the real end-to-end test - self-collision passing (step 5) does **not** guarantee the robot won't collide with *your scene*. Below we plan against cuRobo's bundled `collision_test.yml` scene as a smoke test. This particular scene happens to be tuned around a Franka Panda's geometry, not the RM75-6F's, so a pass here is not a substitute for testing against your actual scene - see the optional diagnostic section below for what to do if planning *does* fail for you.

In [ ]:
from airo_planner import NoPathFoundError
from airo_planner.curobo.single_arm_curobo_planner import SingleArmCuroboPlanner

planner = SingleArmCuroboPlanner(output_path, "collision_test.yml")

q_start = planner.motion_planner.default_joint_state.position.cpu().numpy()
q_goal = q_start.copy()
q_goal[0] += 0.5
q_goal[1] = 0.5
q_goal[3] = 1.6
q_goal[5] = 0.5
q_goal[6] = 0.6


try:
    trajectory = planner.plan_to_joint_configuration(q_start, q_goal)
    print(f"Planned {len(trajectory.path.positions)} waypoints")
except NoPathFoundError:
    print("Planning failed against collision_test.yml - diagnosing below.")

In [ ]:
import time

import torch
from curobo.viewer import ViserVisualizer
from curobo.types import ContentPath, JointState

viz = ViserVisualizer(content_path=ContentPath(robot_config_file=output_path), add_control_frames=True)


def playback(trajectory):
    """Play back a SingleArmTrajectory in the Viser viewer, respecting its timestamps."""
    t = 0.0
    for i, q in enumerate(trajectory.path.positions):
        joint_state = JointState.from_position(
            torch.tensor(q, dtype=torch.float32)[None, :].cuda(), joint_names=planner.motion_planner.joint_names
        ).squeeze(0)
        viz.set_joint_state(joint_state)
        time.sleep(max(0.0, trajectory.times[i] - t))
        t = trajectory.times[i]



In [ ]:
playback(trajectory)


### Optional: diagnosing a failure that self-collision didn't predict

If planning against your scene fails even though `RobotDebugger` (step 5) ruled out self-collision, the next question is whether this is a *world*-collision (fitted spheres bulging into the scene's obstacles) or something else. Isolate it by re-running the identical plan with **no** world obstacles at all - if that succeeds, the fitted spheres themselves are fine in isolation, and the problem is specifically their size/placement relative to this particular scene. This section runs regardless of whether step 6 above passed, as a demonstration of the technique.

In [ ]:
import torch
from curobo.motion_planner import MotionPlanner, MotionPlannerCfg
from curobo.types import JointState

empty_world_config = MotionPlannerCfg.create(robot=output_path, use_cuda_graph=False)  # no scene_model
empty_world_planner = MotionPlanner(empty_world_config)
empty_world_planner.warmup(num_warmup_iterations=1)

start_js = JointState.from_position(
    torch.tensor(q_start, dtype=torch.float32)[None, :].cuda(), joint_names=empty_world_planner.joint_names
)
goal_js = JointState.from_position(
    torch.tensor(q_goal, dtype=torch.float32)[None, :].cuda(), joint_names=empty_world_planner.joint_names
)
result = empty_world_planner.plan_cspace(goal_js, start_js)
print("Plan succeeds with an empty world:", result is not None and bool(result.success.any()))

The empty-world plan above confirms the fitted spheres are collision-free in isolation. If step 6 *did* fail for you, this pinpoints the cause as the sphere fit intersecting the scene's obstacles at the tested poses - a scene/sphere-margin issue, not a bug in the config or the planner.

If it's too conservative for your scene, the fix is iterative and scene-specific:
- Test against *your actual* scene, not an unrelated bundled one like `collision_test.yml`.
- Try lowering `sphere_density` or `clip_links` for the specific link closest to the obstacle in step 2 (use the per-link `coverage`/`protrusion` metrics, or `RobotDebugger`, to judge fit quality - **not** `builder.visualize(show_meshes=True)`, which is unreliable, see step 2), then repeat steps 3-6.
- Re-run this section after any refit - there's no shortcut to re-verifying against your real scene.